# SUNSHIELD — Notebook 1: Data Preprocessing

**Goal:** turn the 4 raw Kaggle CSV files into a clean, scaled, chronologically split dataset
that the LSTM (Notebook 2) can read as sliding windows.

| Step | What happens | Output |
|---|---|---|
| 1 | Load the 4 CSV files and fix the date formats | — |
| 2 | Merge generation + weather for each plant, then combine both plants | merged table |
| 3 | Fill weather gaps only (max 1 hour); never fill `AC_POWER` | missing-value count before/after |
| 4 | Remove night readings (`IRRADIATION == 0`) | rows before/after |
| 5 | Chronological split by days: 70% / 15% / 15% | dates + rows per split |
| 6 | Fit `StandardScaler` on training data only, apply to all splits | `scalers.pkl` |
| 7 | Build sliding windows (3 weather features → `AC_POWER`), skip windows with time gaps | windows per split |
| 8 | Row-count table for every stage | `row_counts.csv` |
| 9 | Save and zip outputs | `step1_outputs.zip` |

**Decisions agreed with Dr. Mai**
- Night = `IRRADIATION == 0`. Night readings are removed, and a window may continue from the evening
  of one day into the morning of the next (the night is skipped, it is *not* counted as a gap).
- Each inverter (`PLANT_ID` + `SOURCE_KEY`) is its own time series. A window never mixes two inverters.

## 0. Settings

In [1]:
import os, shutil, joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Where the 4 CSV files are. Works from the repo (data/raw) and from Colab (files uploaded to /content).
CSV_FILES = ['Plant_1_Generation_Data.csv', 'Plant_1_Weather_Sensor_Data.csv',
             'Plant_2_Generation_Data.csv', 'Plant_2_Weather_Sensor_Data.csv']
DATA_DIR = next(d for d in ['data/raw', '../data/raw', '/content', '.']
                if os.path.exists(os.path.join(d, CSV_FILES[0])))
PROJECT_ROOT = os.path.abspath(os.path.join(DATA_DIR, '../..')) if DATA_DIR.endswith('data/raw') else os.path.abspath(DATA_DIR)
OUT_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'step1_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

FEATURES = ['IRRADIATION', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE']
TARGET = 'AC_POWER'

FREQ = '15min'               # sampling interval of the dataset
MAX_FILL_STEPS = 4           # fill weather gaps of at most 1 hour = 4 readings of 15 min
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15   # test gets the rest (15%)
WINDOW_SIZES = [8, 16, 24, 32, 48]  # candidates for Notebook 2
CROSS_NIGHT = True           # True: a window can go from evening to next morning (night removed)
TARGET_SHIFT = 1             # 1: predict the reading right AFTER the window (next step, as in the report)
                             # 0: the window ends at the predicted reading (uses the current weather too)

print('Data folder  :', os.path.abspath(DATA_DIR))
print('Output folder:', OUT_DIR)

Data folder  : /home/user/SUNSHIELD/data/raw
Output folder: /home/user/SUNSHIELD/outputs/step1_outputs


## Step 1 — Load the 4 CSV files and fix the date formats

The two plants do not use the same date format:
- `Plant_1_Generation_Data.csv` → `15-05-2020 00:00` (day-month-year)
- the other three files → `2020-05-15 00:00:00` (year-month-day)

We give each file its exact format, so pandas never has to guess (guessing can swap day and month).

In [2]:
DATE_FORMATS = {
    'Plant_1_Generation_Data.csv':     '%d-%m-%Y %H:%M',
    'Plant_1_Weather_Sensor_Data.csv': '%Y-%m-%d %H:%M:%S',
    'Plant_2_Generation_Data.csv':     '%Y-%m-%d %H:%M:%S',
    'Plant_2_Weather_Sensor_Data.csv': '%Y-%m-%d %H:%M:%S',
}

raw = {}
for name, fmt in DATE_FORMATS.items():
    df = pd.read_csv(os.path.join(DATA_DIR, name))
    df['DATE_TIME'] = pd.to_datetime(df['DATE_TIME'], format=fmt)
    raw[name] = df

gen1, wea1 = raw['Plant_1_Generation_Data.csv'], raw['Plant_1_Weather_Sensor_Data.csv']
gen2, wea2 = raw['Plant_2_Generation_Data.csv'], raw['Plant_2_Weather_Sensor_Data.csv']

pd.DataFrame([{
    'file': name, 'rows': len(df), 'columns': df.shape[1],
    'first': df['DATE_TIME'].min(), 'last': df['DATE_TIME'].max(),
    'sources (inverters / sensors)': df['SOURCE_KEY'].nunique(),
    'duplicate (time, source)': df.duplicated(['DATE_TIME', 'SOURCE_KEY']).sum(),
    'empty cells': df.isna().sum().sum(),
} for name, df in raw.items()])

,file,rows,columns,first,last,sources (inverters / sensors),"duplicate (time, source)",empty cells
0,Plant_1_Generation_Data.csv,68778,7,2020-05-15,2020-06-17 23:45:00,22,0,0
1,Plant_1_Weather_Sensor_Data.csv,3182,6,2020-05-15,2020-06-17 23:45:00,1,0,0
2,Plant_2_Generation_Data.csv,67698,7,2020-05-15,2020-06-17 23:45:00,22,0,0
3,Plant_2_Weather_Sensor_Data.csv,3259,6,2020-05-15,2020-06-17 23:45:00,1,0,0


## Step 2 — Merge generation + weather for each plant, then combine both plants

- **Generation** files have one row per *inverter* per 15 minutes (22 inverters per plant).
- **Weather** files have one row per *plant* per 15 minutes (one sensor per plant).

So every inverter row gets the weather of its plant at the same `DATE_TIME`.

Some weather timestamps are simply **missing rows** in the weather file. To make them visible we first put the
weather on a complete 15-minute grid (missing timestamps become empty cells), then do a *left* merge, so no
generation row is lost here. Step 3 decides which of these empty cells can be filled.

In [3]:
def weather_on_full_grid(wea):
    # One row per 15 minutes from the first to the last day; missing readings become NaN.
    grid = pd.date_range(wea['DATE_TIME'].min().normalize(),
                         wea['DATE_TIME'].max().normalize() + pd.Timedelta('1D') - pd.Timedelta(FREQ),
                         freq=FREQ)
    out = wea.set_index('DATE_TIME')[FEATURES].reindex(grid)
    out.index.name = 'DATE_TIME'
    out.insert(0, 'PLANT_ID', wea['PLANT_ID'].iloc[0])
    return out.reset_index()

weather_grid = pd.concat([weather_on_full_grid(wea1), weather_on_full_grid(wea2)], ignore_index=True)

merged_parts = []
for gen in (gen1, gen2):
    part = gen[['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'DC_POWER', 'AC_POWER']].merge(
        weather_grid, on=['PLANT_ID', 'DATE_TIME'], how='left')
    merged_parts.append(part)

merged = (pd.concat(merged_parts, ignore_index=True)
            .sort_values(['PLANT_ID', 'SOURCE_KEY', 'DATE_TIME'])
            .reset_index(drop=True))

print('Merged table:', merged.shape)
print('Rows per plant:', merged.groupby('PLANT_ID').size().to_dict())
merged.head()

Merged table: (136476, 8)
Rows per plant: {4135001: 68778, 4136001: 67698}


,DATE_TIME,PLANT_ID,SOURCE_KEY,DC_POWER,AC_POWER,IRRADIATION,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE
0,2020-05-15 00:00:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,25.184316,22.857507
1,2020-05-15 00:15:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,25.084589,22.761668
2,2020-05-15 00:30:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,24.935753,22.592306
3,2020-05-15 00:45:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,24.846130,22.360852
4,2020-05-15 01:00:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,24.621525,22.165423


## Step 3 — Fill weather gaps only (max 1 hour); never fill `AC_POWER`

Rule: a run of missing weather readings is filled by **linear interpolation in time** only if the run is
**at most 4 readings long (≤ 1 hour)** and has a known reading on both sides. Longer gaps stay empty and
those rows are dropped, because inventing several hours of weather would be guessing.

Note: `pandas.interpolate(limit=4)` is *not* enough on its own — on a 7-hour gap it would still fill the first
4 readings. So we measure the length of every gap first and only fill the short ones.

`AC_POWER` is never filled: it is the value we predict and later check for anomalies.
Missing inverter readings stay missing (they become gaps that windows will skip in Step 7).

In [4]:
def fill_short_gaps(series, max_steps):
    is_na = series.isna()
    gap_id = (is_na != is_na.shift()).cumsum()          # a new id every time NaN/non-NaN switches
    gap_len = is_na.groupby(gap_id).transform('sum')    # length of the NaN run each cell belongs to
    filled = series.interpolate(method='linear', limit_area='inside')
    return series.where(~(is_na & (gap_len <= max_steps)), filled)

def describe_gaps(grid):
    rows = []
    for plant, g in grid.groupby('PLANT_ID'):
        is_na = g['IRRADIATION'].isna()
        runs = is_na.groupby((is_na != is_na.shift()).cumsum()).sum()
        runs = runs[runs > 0]
        rows.append({'PLANT_ID': plant, 'missing timestamps': int(is_na.sum()), 'gaps': len(runs),
                     'gaps <= 1 hour': int((runs <= MAX_FILL_STEPS).sum()),
                     'longest gap (readings)': int(runs.max()) if len(runs) else 0})
    return pd.DataFrame(rows)

print('Weather gaps on the 15-minute grid (before filling):')
display(describe_gaps(weather_grid))

n_slots = weather_grid.groupby('PLANT_ID').size()
gen_missing = pd.DataFrame({
    'inverters': merged.groupby('PLANT_ID')['SOURCE_KEY'].nunique(),
    'expected readings': merged.groupby('PLANT_ID')['SOURCE_KEY'].nunique() * n_slots,
    'present': merged.groupby('PLANT_ID').size()})
gen_missing['missing (not filled)'] = gen_missing['expected readings'] - gen_missing['present']
print('Generation (AC_POWER) readings missing per plant — these are never filled:')
display(gen_missing)

missing_before = merged[FEATURES + [TARGET]].isna().sum()

weather_filled = weather_grid.copy()
for col in FEATURES:
    weather_filled[col] = weather_filled.groupby('PLANT_ID')[col].transform(
        lambda s: fill_short_gaps(s, MAX_FILL_STEPS))

# Put the filled weather back into the merged table
merged_filled = merged.drop(columns=FEATURES).merge(weather_filled, on=['PLANT_ID', 'DATE_TIME'], how='left')
missing_after_fill = merged_filled[FEATURES + [TARGET]].isna().sum()

# Rows whose weather is still unknown (gap longer than 1 hour) cannot be used
filled_ok = merged_filled.dropna(subset=FEATURES).reset_index(drop=True)
missing_after_drop = filled_ok[FEATURES + [TARGET]].isna().sum()

missing_values = pd.DataFrame({'before fill': missing_before,
                               'after fill (<= 1 h)': missing_after_fill,
                               'after dropping long gaps': missing_after_drop})
missing_values.index.name = 'column'
print(f'Rows dropped because of weather gaps > 1 hour: {len(merged_filled) - len(filled_ok)}')
missing_values

Weather gaps on the 15-minute grid (before filling):


,PLANT_ID,missing timestamps,gaps,gaps <= 1 hour,longest gap (readings)
0,4135001,82,8,3,28
1,4136001,5,5,5,1


Generation (AC_POWER) readings missing per plant — these are never filled:


,inverters,expected readings,present,missing (not filled)
PLANT_ID,,,,
4135001,22,71808,68778,3030
4136001,22,71808,67698,4110


Rows dropped because of weather gaps > 1 hour: 0


,before fill,after fill (<= 1 h),after dropping long gaps
column,,,
IRRADIATION,4,0,0
AMBIENT_TEMPERATURE,4,0,0
MODULE_TEMPERATURE,4,0,0
AC_POWER,0,0,0


## Step 4 — Remove night readings (`IRRADIATION == 0`)

At night there is no sun, so `AC_POWER = 0` is normal and tells the model nothing. We remove every row with
`IRRADIATION == 0`.

We do **not** remove rows with `AC_POWER == 0` during the day: zero power while the sun is up may be exactly
the kind of fault we want to detect later.

In [5]:
night = filled_ok['IRRADIATION'] == 0
clean = filled_ok[~night].reset_index(drop=True)

day_zero_power = (clean[TARGET] == 0)
print(f'Rows before: {len(filled_ok):,}   night rows removed: {night.sum():,}   rows after: {len(clean):,}')
print(f'Night rows that still had AC_POWER > 0 (removed anyway): {((filled_ok[TARGET] > 0) & night).sum()}')
print(f'Daytime rows with AC_POWER == 0 (kept, possible faults): {day_zero_power.sum():,}')

pd.DataFrame({'before': filled_ok.groupby('PLANT_ID').size(),
              'after': clean.groupby('PLANT_ID').size()})

Rows before: 136,476   night rows removed: 59,374   rows after: 77,102
Night rows that still had AC_POWER > 0 (removed anyway): 14
Daytime rows with AC_POWER == 0 (kept, possible faults): 8,253


,before,after
PLANT_ID,,
4135001,68778,38380
4136001,67698,38722


## Step 5 — Chronological split by days: 70% train / 15% validation / 15% test

Time series must not be shuffled: the model is trained on the past and tested on the future.
We split by **whole days**, so a day is never cut in half. Both plants cover the same dates, so the
same days go to the same split for both plants.

In [6]:
days = np.sort(clean['DATE_TIME'].dt.normalize().unique())
n_days = len(days)
n_train = round(n_days * TRAIN_FRAC)
n_val = round(n_days * VAL_FRAC)
day_split = pd.Series(['train'] * n_train + ['val'] * n_val + ['test'] * (n_days - n_train - n_val),
                      index=days)

clean['SPLIT'] = clean['DATE_TIME'].dt.normalize().map(day_split)
SPLITS = ['train', 'val', 'test']

split_summary = pd.DataFrame([{
    'split': s,
    'first day': day_split[day_split == s].index.min().date(),
    'last day': day_split[day_split == s].index.max().date(),
    'days': int((day_split == s).sum()),
    'rows': int((clean['SPLIT'] == s).sum()),
    'rows %': round(100 * (clean['SPLIT'] == s).mean(), 1),
} for s in SPLITS])
split_summary

,split,first day,last day,days,rows,rows %
0,train,2020-05-15,2020-06-07,24,53342,69.2
1,val,2020-06-08,2020-06-12,5,11968,15.5
2,test,2020-06-13,2020-06-17,5,11792,15.3


## Step 6 — Fit `StandardScaler` on training data only and apply to all splits

$z = (x - \mu) / \sigma$, where $\mu$ and $\sigma$ come **only from the training rows**. Using validation or
test rows here would leak information from the future into training.

Two scalers are saved:
- `features` → the 3 weather inputs
- `target` → `AC_POWER` (Notebook 2 uses `inverse_transform` to bring predictions back to kW)

The original (unscaled) columns are kept next to the scaled ones (suffix `_z`).

In [7]:
FEATURES_Z = [c + '_z' for c in FEATURES]
TARGET_Z = TARGET + '_z'

train_rows = clean['SPLIT'] == 'train'
feature_scaler = StandardScaler().fit(clean.loc[train_rows, FEATURES])
target_scaler = StandardScaler().fit(clean.loc[train_rows, [TARGET]])

clean[FEATURES_Z] = feature_scaler.transform(clean[FEATURES])
clean[TARGET_Z] = target_scaler.transform(clean[[TARGET]]).ravel()

joblib.dump({'features': feature_scaler, 'target': target_scaler,
             'feature_names': FEATURES, 'target_name': TARGET},
            os.path.join(OUT_DIR, 'scalers.pkl'))

print('Scaler statistics (training data only):')
display(pd.DataFrame({'mean': np.r_[feature_scaler.mean_, target_scaler.mean_],
                      'std': np.r_[feature_scaler.scale_, target_scaler.scale_]},
                     index=FEATURES + [TARGET]).round(3))
print('Check — mean / std of the scaled columns per split (train should be 0 / 1):')
clean.groupby('SPLIT')[FEATURES_Z + [TARGET_Z]].agg(['mean', 'std']).round(2).loc[SPLITS]

Scaler statistics (training data only):


,mean,std
IRRADIATION,0.432,0.316
AMBIENT_TEMPERATURE,29.245,4.056
MODULE_TEMPERATURE,40.391,11.813
AC_POWER,506.746,404.999


Check — mean / std of the scaled columns per split (train should be 0 / 1):


IRRADIATION_z       AMBIENT_TEMPERATURE_z       MODULE_TEMPERATURE_z  \
               mean   std                  mean   std                 mean   
SPLIT                                                                        
train          0.00  1.00                  0.00  1.00                -0.00   
val           -0.23  0.88                 -0.42  0.80                -0.36   
test          -0.25  0.83                 -0.57  0.62                -0.43   

            AC_POWER_z        
        std       mean   std  
SPLIT                         
train  1.00      -0.00  1.00  
val    0.85      -0.16  0.88  
test   0.73      -0.16  0.86

## Step 7 — Build sliding windows (3 weather features → `AC_POWER`), skip windows with time gaps

A window is `W` consecutive readings of the 3 scaled weather features; its label is the scaled `AC_POWER`
of the reading right after the window (`TARGET_SHIFT = 1`).

**What counts as consecutive?** For each plant we list, in time order, every 15-minute timestamp that is
*not* night. Two readings of the same inverter are consecutive if they are neighbours in that list.
- the night between two days is skipped, so evening → next morning is still consecutive (`CROSS_NIGHT = True`)
- a missing inverter reading or a weather gap longer than 1 hour breaks the sequence → windows that would
  cross it are skipped

Each unbroken stretch of readings gets a `RUN_ID`. Windows are only taken inside one run, so a window
never crosses a gap and never mixes two inverters.

A window belongs to the split of its **target** reading. The first windows of validation/test may take
their *inputs* (weather only) from the last hours of the previous split; the value being predicted is always
inside its own split.

In [8]:
# Position of every non-night timestamp, per plant (unknown weather counts as non-night so it breaks runs)
timeline = weather_filled[weather_filled['IRRADIATION'] != 0][['PLANT_ID', 'DATE_TIME']].copy()
timeline['POS'] = timeline.groupby('PLANT_ID').cumcount()
clean = clean.merge(timeline, on=['PLANT_ID', 'DATE_TIME'], how='left')
clean = clean.sort_values(['PLANT_ID', 'SOURCE_KEY', 'DATE_TIME']).reset_index(drop=True)

same_series = (clean['PLANT_ID'].eq(clean['PLANT_ID'].shift())
               & clean['SOURCE_KEY'].eq(clean['SOURCE_KEY'].shift()))
next_in_time = clean['POS'].diff().eq(1)
if not CROSS_NIGHT:
    next_in_time &= clean['DATE_TIME'].dt.normalize().eq(clean['DATE_TIME'].dt.normalize().shift())
clean['RUN_ID'] = (~(same_series & next_in_time)).cumsum() - 1

run_len = clean.groupby('RUN_ID').size()
print(f'{clean["RUN_ID"].nunique():,} unbroken runs over {clean.groupby(["PLANT_ID", "SOURCE_KEY"]).ngroups} '
      f'inverters | run length: median {int(run_len.median())}, max {run_len.max()} readings')

415 unbroken runs over 44 inverters | run length: median 102, max 791 readings


In [9]:
def build_windows(df, window, target_shift=TARGET_SHIFT, features=FEATURES_Z, target=TARGET_Z):
    '''Returns {split: (X, y, meta)}.
    X: (n, window, n_features) scaled inputs, y: (n,) scaled AC_POWER,
    meta: PLANT_ID, SOURCE_KEY, DATE_TIME and AC_POWER (kW) of each target reading.
    df must be sorted by inverter and time and have RUN_ID and SPLIT columns.'''
    X_parts, target_rows = [], []
    feats = df[features].to_numpy(np.float32)
    for rows in df.groupby('RUN_ID', sort=False).indices.values():
        n_win = len(rows) - window - target_shift + 1
        if n_win <= 0:
            continue
        w = np.lib.stride_tricks.sliding_window_view(feats[rows], window, axis=0)[:n_win]
        X_parts.append(w.transpose(0, 2, 1))
        target_rows.append(rows[window - 1 + target_shift:])
    X = np.concatenate(X_parts)
    t = np.concatenate(target_rows)
    y = df[target].to_numpy(np.float32)[t]
    meta = df.iloc[t][['PLANT_ID', 'SOURCE_KEY', 'DATE_TIME', TARGET, 'SPLIT']].reset_index(drop=True)
    return {s: (X[(meta['SPLIT'] == s).to_numpy()], y[(meta['SPLIT'] == s).to_numpy()],
                meta[meta['SPLIT'] == s].drop(columns='SPLIT').reset_index(drop=True))
            for s in SPLITS}

windows_per_split = pd.DataFrame(
    {W: {s: len(v[1]) for s, v in build_windows(clean, W).items()} for W in WINDOW_SIZES}).T
windows_per_split.index.name = 'window size'
windows_per_split['total'] = windows_per_split.sum(axis=1)
windows_per_split['hours of history'] = [W * 15 / 60 for W in WINDOW_SIZES]
windows_per_split

,train,val,test,total,hours of history
window size,,,,,
8,50249,11968,11616,73833,2.0
16,47289,11968,11440,70697,4.0
24,44401,11968,11264,67633,6.0
32,41589,11968,11088,64645,8.0
48,36157,11968,10736,58861,12.0


In [10]:
# Example: shapes for one window size + sanity checks
example = build_windows(clean, 24)
for s, (X, y, meta) in example.items():
    print(f'{s:5s}  X {X.shape}  y {y.shape}  targets from {meta.DATE_TIME.min()} to {meta.DATE_TIME.max()}')

X, y, meta = example['train']
assert not np.isnan(X).any() and not np.isnan(y).any()
assert np.allclose(target_scaler.inverse_transform(y[:5, None]).ravel(), meta[TARGET].head(5), atol=1e-2)
print('No NaN in windows; y converts back to the original kW values.')

train  X (44401, 24, 3)  y (44401,)  targets from 2020-05-15 11:45:00 to 2020-06-07 19:00:00
val    X (11968, 24, 3)  y (11968,)  targets from 2020-06-08 05:45:00 to 2020-06-12 18:45:00
test   X (11264, 24, 3)  y (11264,)  targets from 2020-06-13 05:45:00 to 2020-06-17 18:45:00
No NaN in windows; y converts back to the original kW values.


## Step 8 — Row-count table for every stage (raw → merged → cleaned → windows)

In [11]:
def per_plant(df):
    c = df.groupby('PLANT_ID').size()
    return [int(c.get(p, 0)) for p in sorted(merged['PLANT_ID'].unique())] + [len(df)]

stages = [
    ('1. raw generation rows', pd.concat([gen1, gen2])),
    ('2. merged with weather', merged),
    ('3. after weather fill (dropped gaps > 1 h)', filled_ok),
    ('4. cleaned (night removed)', clean),
]
row_counts = pd.DataFrame([[name] + per_plant(df) for name, df in stages],
                          columns=['stage', 'plant_1', 'plant_2', 'total'])
for W in WINDOW_SIZES:
    meta_all = pd.concat([v[2] for v in build_windows(clean, W).values()])
    row_counts.loc[len(row_counts)] = [f'5. windows (W={W})'] + per_plant(meta_all)
row_counts

,stage,plant_1,plant_2,total
0,1. raw generation rows,68778,67698,136476
1,2. merged with weather,68778,67698,136476
2,3. after weather fill (dropped gaps > 1 h),68778,67698,136476
3,4. cleaned (night removed),38380,38722,77102
4,5. windows (W=8),36071,37762,73833
5,5. windows (W=16),33895,36802,70697
6,5. windows (W=24),31751,35882,67633
7,5. windows (W=32),29611,35034,64645
8,5. windows (W=48),25523,33338,58861


## Step 9 — Save and zip outputs

| File | Content |
|---|---|
| `merged_data.csv` | Step 2 table (all rows, weather not filled yet) |
| `clean_data.csv` | Final table: raw + scaled (`_z`) columns, `SPLIT`, `RUN_ID` — input of Notebook 2 |
| `scalers.pkl` | feature + target `StandardScaler` (fitted on train only) |
| `missing_values.csv` | Step 3 before/after counts |
| `split_summary.csv` | Step 5 dates and rows per split |
| `windows_per_split.csv` | Step 7 window counts for every candidate size |
| `row_counts.csv` | Step 8 table |

To rebuild the windows in Notebook 2: load `clean_data.csv`, sort by `PLANT_ID, SOURCE_KEY, DATE_TIME`,
and reuse the `build_windows` function above.

In [12]:
merged.to_csv(os.path.join(OUT_DIR, 'merged_data.csv'), index=False)
clean_cols = (['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'SPLIT', 'RUN_ID', TARGET] + FEATURES
              + [TARGET_Z] + FEATURES_Z)
clean[clean_cols].to_csv(os.path.join(OUT_DIR, 'clean_data.csv'), index=False)
missing_values.to_csv(os.path.join(OUT_DIR, 'missing_values.csv'))
split_summary.to_csv(os.path.join(OUT_DIR, 'split_summary.csv'), index=False)
windows_per_split.to_csv(os.path.join(OUT_DIR, 'windows_per_split.csv'))
row_counts.to_csv(os.path.join(OUT_DIR, 'row_counts.csv'), index=False)

zip_path = shutil.make_archive(os.path.join(PROJECT_ROOT, 'outputs', 'step1_outputs'), 'zip', OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    print(f'{f:25s} {os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6:6.2f} MB')
print('Zipped to', zip_path)

try:                                  # download automatically when running on Google Colab
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass

clean_data.csv             15.08 MB
merged_data.csv            14.55 MB
missing_values.csv          0.00 MB
row_counts.csv              0.00 MB
scalers.pkl                 0.00 MB
split_summary.csv           0.00 MB
windows_per_split.csv       0.00 MB
Zipped to /home/user/SUNSHIELD/outputs/step1_outputs.zip
